# NB09 — Statistical Robustness, Calibration, and Selective Classification

This notebook strengthens the inferential and probabilistic evaluation of the dry-bean classification study **without retraining the base classifiers**.

## Objectives
1. Harmonize the leakage-safe out-of-fold (OOF) predictions from NB06 and NB08.
2. Compare FULL_11 models with a **Nadeau–Bengio corrected resampled t-test** using the matched 3 × 5 repeated-CV folds.
3. Perform **equivalence-margin sensitivity analysis** (±0.005, ±0.010, ±0.015 macro-F1) using corrected standard errors.
4. Report a correlated-t probability summary with ROPE sensitivity.
5. Quantify probabilistic performance using multiclass Brier score, negative log-likelihood, top-label ECE, and MCE.
6. Apply **cross-fitted temperature scaling** to OOF probabilities. Each evaluated outer fold is calibrated only with OOF predictions from the other four folds of the same seed.
7. Produce reliability diagrams and raw-vs-calibrated summaries.
8. Evaluate **selective classification** through accuracy–coverage curves using fold-external confidence thresholds: for each held-out outer fold, both the temperature parameter and the confidence threshold are estimated exclusively from the other four outer folds of the same seed.

> **Important methodological note.** Repeated-CV folds are not treated as independent observations. Corrected-resampling inference is used for fold-level comparisons. OOF rows repeated across seeds are model-evaluation predictions, not additional independent biological grains. The primary practical-equivalence margin is prespecified at ±0.010 macro-F1; ±0.005 and ±0.015 are retained as sensitivity bounds.


> **Execution:** this notebook is designed for **Run all**. It reuses saved OOF predictions and does not retrain base models. **CPU is sufficient**. Expected runtime in Colab is roughly **10–20 minutes**, with cross-fitted temperature scaling being the slowest section.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, sys, json, platform, warnings, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import minimize_scalar
from scipy.stats import t as student_t
from statsmodels.stats.multitest import multipletests
from sklearn.metrics import (
    f1_score, accuracy_score, log_loss
)

warnings.filterwarnings('ignore')

ROOT = Path('/content/drive/MyDrive/DRY_BEAN_HYBRID_Q1')
RESULTS = ROOT/'03_RESULTS'
FIGURES = ROOT/'05_FIGURES'
TABLES = ROOT/'06_TABLES'
LOGS = ROOT/'07_LOGS'

OUT = RESULTS/'NB09_ROBUSTNESS_CALIBRATION_SELECTIVE'
FIG = FIGURES/'NB09_ROBUSTNESS_CALIBRATION_SELECTIVE'
TAB = TABLES/'NB09_ROBUSTNESS_CALIBRATION_SELECTIVE'
LOG = LOGS/'NB09_ROBUSTNESS_CALIBRATION_SELECTIVE'

for p in [OUT, FIG, TAB, LOG]:
    p.mkdir(parents=True, exist_ok=True)

NB06_METRICS = RESULTS/'NB06_STATS'/'all_model_metrics_by_fold.csv'
NB06_OOF = RESULTS/'NB06_STATS'/'all_oof_predictions.csv'
NB08_METRICS = RESULTS/'NB08_EXTENDED_ABLATION'/'nb08_metrics_by_fold.csv'
NB08_OOF = RESULTS/'NB08_EXTENDED_ABLATION'/'nb08_oof_predictions.csv'

for p in [NB06_METRICS, NB06_OOF, NB08_METRICS, NB08_OOF]:
    assert p.exists(), f'Missing required input: {p}'

SEEDS = [2026, 2027, 2028]
OUTER_FOLDS = 5
N_TRAIN = 3200
N_TEST = 800
TEST_TRAIN_RATIO = N_TEST / N_TRAIN

ALPHA = 0.05
PRIMARY_EQUIV_MARGIN = 0.010
EQUIV_MARGINS = [0.005, 0.010, 0.015]  # primary ±0.010; ±0.005 and ±0.015 sensitivity bounds
ROPE_MARGINS = [0.005, 0.010]
ECE_BINS = 10
TOP_N_PLOTS = 6
RUN_TEMPERATURE_SCALING = True

print('NB09 output:', OUT)
print('Base-model retraining: NO')
print('Temperature scaling:', RUN_TEMPERATURE_SCALING)


## 1. Load and harmonize OOF predictions

NB06 contains the original FULL_11 models. NB08 contributes the additional FULL_11 comparators and all REDUCED_7 reruns. The combined table is checked so that each model × feature-set × seed has exactly 4000 OOF predictions.


In [ ]:
m6 = pd.read_csv(NB06_METRICS)
o6 = pd.read_csv(NB06_OOF)
m8 = pd.read_csv(NB08_METRICS)
o8 = pd.read_csv(NB08_OOF)

m6['feature_set'] = 'FULL_11'
o6['feature_set'] = 'FULL_11'

metric_cols = [
    'f1_macro','accuracy','balanced_accuracy','precision_macro','recall_macro',
    'mcc','kappa','log_loss','feature_set','model','seed','outer_fold'
]
metrics = pd.concat([
    m6[metric_cols],
    m8[metric_cols]
], ignore_index=True)

oof = pd.concat([o6, o8], ignore_index=True, sort=False)

# Remove accidental duplicate records if a future upstream notebook contains overlap.
key_metric = ['feature_set','model','seed','outer_fold']
metrics = metrics.sort_values(key_metric).drop_duplicates(key_metric, keep='last').reset_index(drop=True)

key_oof = ['feature_set','model','seed','outer_fold','row_id']
oof = oof.sort_values(key_oof).drop_duplicates(key_oof, keep='last').reset_index(drop=True)

prob_cols = [c for c in oof.columns if c.startswith('prob_')]
CLASSES = [c.replace('prob_','',1) for c in prob_cols]

assert len(CLASSES) == 4, CLASSES
assert set(oof['y_true'].unique()) == set(CLASSES)

counts = oof.groupby(['feature_set','model','seed']).size()
assert (counts == 4000).all(), counts[counts != 4000]

fold_counts = oof.groupby(['feature_set','model','seed','outer_fold']).size()
assert (fold_counts == 800).all(), fold_counts[fold_counts != 800]

metrics.to_csv(OUT/'all_metrics_harmonized.csv', index=False)
oof.to_csv(OUT/'all_oof_harmonized.csv', index=False)

print('Classes:', CLASSES)
print('FULL_11 models:', metrics.loc[metrics.feature_set=='FULL_11','model'].nunique())
print('REDUCED_7 models:', metrics.loc[metrics.feature_set=='REDUCED_7','model'].nunique())
print('OOF rows:', len(oof))


## 2. FULL_11 ranking and automatic reference model

The reference model for inferential comparisons is selected **only by mean macro-F1 across the 15 matched outer-test folds**. The ranking is descriptive; statistical comparisons are reported separately below.


In [ ]:
full_metrics = metrics[metrics.feature_set=='FULL_11'].copy()

rank_full = (
    full_metrics.groupby('model')
    .agg(
        mean_macro_f1=('f1_macro','mean'),
        sd_macro_f1=('f1_macro','std'),
        mean_accuracy=('accuracy','mean'),
        mean_balanced_accuracy=('balanced_accuracy','mean'),
        mean_log_loss=('log_loss','mean'),
        n_folds=('f1_macro','size')
    )
    .reset_index()
    .sort_values(['mean_macro_f1','mean_accuracy'], ascending=False)
    .reset_index(drop=True)
)
rank_full['rank_macro_f1'] = np.arange(1, len(rank_full)+1)
rank_full.to_csv(TAB/'full11_model_ranking.csv', index=False)

REFERENCE_MODEL = rank_full.loc[0,'model']
print('Reference model:', REFERENCE_MODEL)
display(rank_full)


## 3. Corrected-resampling inference for FULL_11

For each comparator, fold-level macro-F1 differences are paired to the same seed and outer fold as the reference model. The corrected standard error is

\[
SE_{corr}=\sqrt{\left(\frac{1}{n}+\frac{n_{test}}{n_{train}}\right)s_d^2},
\]

with \(n=15\), \(n_{test}/n_{train}=800/3200=0.25\). Two-sided p-values are Holm-adjusted across reference-vs-comparator tests.


In [ ]:
def corrected_stats(d, test_train_ratio=TEST_TRAIN_RATIO, alpha=ALPHA):
    d = np.asarray(d, dtype=float)
    n = len(d)
    mean_d = float(np.mean(d))
    sd_d = float(np.std(d, ddof=1))
    var_d = sd_d**2
    se_corr = math.sqrt((1/n + test_train_ratio) * var_d)
    df = n - 1
    if se_corr == 0:
        t_stat = np.inf if mean_d > 0 else (-np.inf if mean_d < 0 else 0.0)
        p_two = 0.0 if mean_d != 0 else 1.0
        ci_low = ci_high = mean_d
    else:
        t_stat = mean_d / se_corr
        p_two = 2 * student_t.sf(abs(t_stat), df)
        crit = student_t.ppf(1-alpha/2, df)
        ci_low = mean_d - crit*se_corr
        ci_high = mean_d + crit*se_corr
    return {
        'n_pairs': n,
        'mean_delta_macro_f1': mean_d,
        'sd_delta': sd_d,
        'corrected_se': se_corr,
        'df': df,
        't_corrected': t_stat,
        'p_two_sided': p_two,
        'ci95_low': ci_low,
        'ci95_high': ci_high
    }

ref = (
    full_metrics[full_metrics.model==REFERENCE_MODEL]
    [['seed','outer_fold','f1_macro']]
    .rename(columns={'f1_macro':'f1_ref'})
)

rows = []
for model in rank_full.model:
    if model == REFERENCE_MODEL:
        continue
    cur = (
        full_metrics[full_metrics.model==model]
        [['seed','outer_fold','f1_macro']]
        .rename(columns={'f1_macro':'f1_model'})
    )
    z = ref.merge(cur, on=['seed','outer_fold'], validate='one_to_one')
    assert len(z) == 15
    # Positive delta means comparator > reference.
    d = z.f1_model.values - z.f1_ref.values
    st = corrected_stats(d)
    st['reference_model'] = REFERENCE_MODEL
    st['comparator_model'] = model
    rows.append(st)

corr_tests = pd.DataFrame(rows)
reject, p_holm, _, _ = multipletests(corr_tests.p_two_sided.values, alpha=ALPHA, method='holm')
corr_tests['p_holm'] = p_holm
corr_tests['significant_holm_0.05'] = reject
corr_tests = corr_tests.sort_values('mean_delta_macro_f1', ascending=False).reset_index(drop=True)
corr_tests.to_csv(TAB/'full11_corrected_resampled_tests_vs_reference.csv', index=False)

display(corr_tests)


## 4. Equivalence-margin and correlated-t probability sensitivity

Equivalence is evaluated as a **sensitivity analysis**, not as an arbitrary declaration of practical equivalence. For each pre-specified margin, both one-sided tests must reject at \(lpha=0.05\) using the corrected standard error.

A correlated-t probability summary is also reported for ROPEs of ±0.005 and ±0.010 macro-F1. These probabilities summarize the corrected-t uncertainty model and should not be interpreted as independent replication probabilities.


In [ ]:
def corrected_tost(mean_d, se, df, margin):
    if se == 0:
        equiv = abs(mean_d) < margin
        return 0.0 if equiv else 1.0, 0.0 if equiv else 1.0, 0.0 if equiv else 1.0, equiv
    t_lower = (mean_d + margin) / se   # H0: delta <= -margin
    p_lower = student_t.sf(t_lower, df)
    t_upper = (mean_d - margin) / se   # H0: delta >= +margin
    p_upper = student_t.cdf(t_upper, df)
    p_tost = max(p_lower, p_upper)
    return p_lower, p_upper, p_tost, bool(p_tost < ALPHA)

def correlated_t_probs(mean_d, se, df, rope):
    if se == 0:
        return (
            float(mean_d < -rope),
            float(abs(mean_d) <= rope),
            float(mean_d > rope)
        )
    p_below = student_t.cdf((-rope - mean_d)/se, df)
    p_above = student_t.sf((rope - mean_d)/se, df)
    p_rope = max(0.0, 1.0 - p_below - p_above)
    return p_below, p_rope, p_above

eq_rows = []
prob_rows = []

for _, r in corr_tests.iterrows():
    for margin in EQUIV_MARGINS:
        p1,p2,pt,equiv = corrected_tost(
            r.mean_delta_macro_f1, r.corrected_se, int(r.df), margin
        )
        eq_rows.append({
            'reference_model': REFERENCE_MODEL,
            'comparator_model': r.comparator_model,
            'mean_delta_comparator_minus_reference': r.mean_delta_macro_f1,
            'margin': margin,
            'p_lower': p1,
            'p_upper': p2,
            'p_tost': pt,
            'equivalent_at_alpha_0.05': equiv
        })
    for rope in ROPE_MARGINS:
        pb,pr,pa = correlated_t_probs(
            r.mean_delta_macro_f1, r.corrected_se, int(r.df), rope
        )
        prob_rows.append({
            'reference_model': REFERENCE_MODEL,
            'comparator_model': r.comparator_model,
            'mean_delta_comparator_minus_reference': r.mean_delta_macro_f1,
            'rope': rope,
            'p_comparator_worse_than_rope': pb,
            'p_practically_equivalent_within_rope': pr,
            'p_comparator_better_than_rope': pa
        })

equiv = pd.DataFrame(eq_rows)
corr_prob = pd.DataFrame(prob_rows)

equiv.to_csv(TAB/'full11_equivalence_margin_sensitivity.csv', index=False)
corr_prob.to_csv(TAB/'full11_correlated_t_probability_summary.csv', index=False)

print('Equivalence sensitivity:')
display(equiv)
print('Correlated-t probability summary:')
display(corr_prob)


## 5. Raw probabilistic calibration metrics

Reported metrics:
- **Multiclass Brier score**: mean summed squared probability error across the four classes.
- **NLL**: multiclass log loss.
- **Top-label ECE**: 10-bin expected calibration error using maximum predicted probability and correctness.
- **MCE**: maximum absolute calibration gap among populated bins.

These are computed for each model × feature set × seed from leakage-safe OOF predictions.


In [ ]:
def normalize_probs(P, eps=1e-12):
    P = np.asarray(P, dtype=float)
    P = np.clip(P, eps, 1-eps)
    P = P / P.sum(axis=1, keepdims=True)
    return P

def multiclass_brier(y_true, P, classes=CLASSES):
    y_true = np.asarray(y_true)
    P = normalize_probs(P)
    Y = np.column_stack([(y_true == c).astype(float) for c in classes])
    return float(np.mean(np.sum((P-Y)**2, axis=1)))

def top_label_calibration(y_true, P, classes=CLASSES, n_bins=ECE_BINS):
    P = normalize_probs(P)
    pred_idx = np.argmax(P, axis=1)
    pred = np.asarray(classes)[pred_idx]
    conf = np.max(P, axis=1)
    correct = (pred == np.asarray(y_true)).astype(float)
    edges = np.linspace(0,1,n_bins+1)
    rows=[]
    ece=0.0
    mce=0.0
    for b in range(n_bins):
        lo, hi = edges[b], edges[b+1]
        mask = (conf >= lo) & ((conf < hi) if b < n_bins-1 else (conf <= hi))
        n = int(mask.sum())
        if n == 0:
            continue
        acc = float(correct[mask].mean())
        cbar = float(conf[mask].mean())
        gap = abs(acc-cbar)
        ece += (n/len(conf))*gap
        mce = max(mce, gap)
        rows.append({
            'bin': b+1, 'lower': lo, 'upper': hi, 'n': n,
            'accuracy': acc, 'mean_confidence': cbar, 'gap': gap
        })
    return float(ece), float(mce), pd.DataFrame(rows)

def probability_metrics(g, prob_columns=prob_cols):
    P = normalize_probs(g[prob_columns].to_numpy())
    yt = g.y_true.to_numpy()
    pred = np.asarray(CLASSES)[np.argmax(P,axis=1)]
    ece,mce,_ = top_label_calibration(yt,P)
    return {
        'accuracy': accuracy_score(yt,pred),
        'macro_f1': f1_score(yt,pred,average='macro'),
        'nll': log_loss(yt,P,labels=CLASSES),
        'brier_multiclass': multiclass_brier(yt,P),
        'ece_toplabel_10bin': ece,
        'mce_toplabel_10bin': mce
    }

raw_rows=[]
for (fs,model,seed),g in oof.groupby(['feature_set','model','seed'], sort=True):
    z=probability_metrics(g)
    z.update({'feature_set':fs,'model':model,'seed':seed,'calibration':'raw'})
    raw_rows.append(z)

raw_cal = pd.DataFrame(raw_rows)
raw_cal.to_csv(OUT/'raw_probability_metrics_by_seed.csv',index=False)

raw_summary = (
    raw_cal.groupby(['feature_set','model'])
    .agg(
        macro_f1_mean=('macro_f1','mean'),
        brier_mean=('brier_multiclass','mean'),
        nll_mean=('nll','mean'),
        ece_mean=('ece_toplabel_10bin','mean'),
        ece_sd=('ece_toplabel_10bin','std'),
        mce_mean=('mce_toplabel_10bin','mean')
    )
    .reset_index()
    .sort_values(['feature_set','brier_mean'])
)
raw_summary.to_csv(TAB/'raw_calibration_summary.csv',index=False)
display(raw_summary)


## 6. Cross-fitted temperature scaling

For each model, feature set, and seed:
- hold one outer fold out;
- estimate a single temperature \(T>0\) from OOF probabilities of the other four folds;
- apply that temperature to the held-out fold;
- repeat for all five folds.

This adds a leakage-safe post-hoc probability-calibration layer **without retraining the base classifier** and without using the evaluated fold to estimate its temperature.


In [ ]:
def temperature_transform(P, T, eps=1e-12):
    P = normalize_probs(P, eps=eps)
    logits = np.log(np.clip(P, eps, 1.0))
    logits = logits / float(T)
    logits = logits - logits.max(axis=1, keepdims=True)
    expz = np.exp(logits)
    return expz / expz.sum(axis=1, keepdims=True)

def fit_temperature(y_true, P):
    P = normalize_probs(P)
    y_true = np.asarray(y_true)
    def objective(logT):
        T = float(np.exp(logT))
        PT = temperature_transform(P,T)
        return log_loss(y_true,PT,labels=CLASSES)
    res = minimize_scalar(
        objective,
        bounds=(math.log(0.05), math.log(20.0)),
        method='bounded',
        options={'xatol':1e-5}
    )
    return float(np.exp(res.x)), float(res.fun)

calibrated_parts=[]
temp_rows=[]

if RUN_TEMPERATURE_SCALING:
    for (fs,model,seed),g0 in oof.groupby(['feature_set','model','seed'], sort=True):
        g0=g0.copy()
        for fold in sorted(g0.outer_fold.unique()):
            train_cal = g0[g0.outer_fold != fold]
            test_cal = g0[g0.outer_fold == fold].copy()
            T, train_nll = fit_temperature(
                train_cal.y_true.values,
                train_cal[prob_cols].to_numpy()
            )
            Ptest = temperature_transform(test_cal[prob_cols].to_numpy(),T)
            for j,c in enumerate(prob_cols):
                test_cal['cal_'+c] = Ptest[:,j]
            temp_rows.append({
                'feature_set':fs,'model':model,'seed':int(seed),
                'outer_fold':int(fold),'temperature':T,
                'calibration_train_nll':train_nll
            })
            calibrated_parts.append(test_cal)

    cal_oof = pd.concat(calibrated_parts,ignore_index=True)
    temp_df = pd.DataFrame(temp_rows)
    temp_df.to_csv(TAB/'crossfit_temperature_by_fold.csv',index=False)
    cal_oof.to_csv(OUT/'crossfit_temperature_scaled_oof.csv',index=False)
    print('Cross-fitted calibration rows:',len(cal_oof))
    display(temp_df.groupby(['feature_set','model']).temperature.agg(['mean','std','min','max']).reset_index())
else:
    cal_oof = None
    temp_df = pd.DataFrame()
    print('Temperature scaling disabled.')


## 7. Raw vs calibrated probability performance

Accuracy and class predictions are expected to remain unchanged under scalar temperature scaling because the transformation preserves the probability ordering. The main quantities of interest are NLL, Brier score, ECE, and MCE.


In [ ]:
cal_rows=[]
if RUN_TEMPERATURE_SCALING:
    cal_prob_cols=['cal_'+c for c in prob_cols]
    for (fs,model,seed),g in cal_oof.groupby(['feature_set','model','seed'], sort=True):
        P = normalize_probs(g[cal_prob_cols].to_numpy())
        yt=g.y_true.to_numpy()
        pred=np.asarray(CLASSES)[np.argmax(P,axis=1)]
        ece,mce,_=top_label_calibration(yt,P)
        cal_rows.append({
            'feature_set':fs,'model':model,'seed':seed,'calibration':'temperature_crossfit',
            'accuracy':accuracy_score(yt,pred),
            'macro_f1':f1_score(yt,pred,average='macro'),
            'nll':log_loss(yt,P,labels=CLASSES),
            'brier_multiclass':multiclass_brier(yt,P),
            'ece_toplabel_10bin':ece,
            'mce_toplabel_10bin':mce
        })

    cal_metrics=pd.DataFrame(cal_rows)
    cal_metrics.to_csv(OUT/'temperature_scaled_probability_metrics_by_seed.csv',index=False)

    both=pd.concat([raw_cal,cal_metrics],ignore_index=True)
    cal_compare=(
        both.groupby(['feature_set','model','calibration'])
        .agg(
            macro_f1_mean=('macro_f1','mean'),
            accuracy_mean=('accuracy','mean'),
            brier_mean=('brier_multiclass','mean'),
            nll_mean=('nll','mean'),
            ece_mean=('ece_toplabel_10bin','mean'),
            mce_mean=('mce_toplabel_10bin','mean')
        ).reset_index()
    )
    cal_compare.to_csv(TAB/'raw_vs_temperature_calibration_summary.csv',index=False)

    wide=cal_compare.pivot(index=['feature_set','model'],columns='calibration')
    display(cal_compare.sort_values(['feature_set','model','calibration']))
else:
    cal_metrics=pd.DataFrame()
    cal_compare=pd.DataFrame()


## 8. Reliability diagrams for the leading FULL_11 models

The plotted models are selected automatically as the top `TOP_N_PLOTS` FULL_11 models by mean macro-F1. Raw and cross-fitted temperature-scaled reliability curves are shown separately for each model.


In [ ]:
TOP_MODELS = rank_full.head(TOP_N_PLOTS).model.tolist()
print('Reliability models:',TOP_MODELS)

def aggregate_reliability(g, prob_columns, n_bins=ECE_BINS):
    P=normalize_probs(g[prob_columns].to_numpy())
    _,_,bins=top_label_calibration(g.y_true.to_numpy(),P,n_bins=n_bins)
    return bins

for model in TOP_MODELS:
    graw=oof[(oof.feature_set=='FULL_11') & (oof.model==model)]
    bins_raw=aggregate_reliability(graw,prob_cols)

    plt.figure(figsize=(6,6))
    plt.plot([0,1],[0,1],'--',label='Perfect calibration')
    plt.plot(bins_raw.mean_confidence,bins_raw.accuracy,marker='o',label='Raw')

    if RUN_TEMPERATURE_SCALING:
        gcal=cal_oof[(cal_oof.feature_set=='FULL_11') & (cal_oof.model==model)]
        bins_cal=aggregate_reliability(gcal,['cal_'+c for c in prob_cols])
        plt.plot(bins_cal.mean_confidence,bins_cal.accuracy,marker='o',label='Cross-fit temperature')

    plt.xlabel('Mean confidence')
    plt.ylabel('Empirical accuracy')
    plt.title(f'Reliability diagram — {model} (FULL_11)')
    plt.xlim(0,1)
    plt.ylim(0,1)
    plt.legend()
    plt.tight_layout()
    safe=''.join(ch if ch.isalnum() or ch in '-_' else '_' for ch in model)
    plt.savefig(FIG/f'reliability_FULL11_{safe}.png',dpi=300,bbox_inches='tight')
    plt.show()


## 9. Selective classification: fold-external accuracy–coverage

For each seed and held-out outer fold, the confidence rule is selected **without using that fold**. Temperature scaling is first fitted on OOF predictions from the other four outer folds of the same seed. The same four-fold design set is then used to determine the confidence threshold corresponding to each prespecified target coverage. The fitted temperature and threshold are finally applied to the untouched held-out outer fold.

Target coverages are 100%, 95%, 90%, 80%, 70%, 60%, and 50%. Because the threshold is estimated outside the evaluated fold, realized held-out coverage can differ slightly from the nominal target. This analysis quantifies the performance–coverage trade-off without choosing a threshold from the cases on which accepted-case accuracy is evaluated.


In [ ]:
# ============================================================
# Fold-external selective classification
# For each held-out outer fold:
#   1) fit temperature on the other four OOF folds of the same seed;
#   2) choose the confidence threshold on those same four folds;
#   3) apply temperature + threshold to the untouched held-out fold.
# ============================================================
TARGET_COVERAGES = [1.00, 0.95, 0.90, 0.80, 0.70, 0.60, 0.50]

fold_rows = []
accepted_parts = []

for (fs, model, seed), g0 in oof.groupby(['feature_set', 'model', 'seed'], sort=True):
    g0 = g0.copy()

    for fold in sorted(g0.outer_fold.unique()):
        design = g0[g0.outer_fold != fold].copy()
        heldout = g0[g0.outer_fold == fold].copy()

        # Calibration fitted without the held-out fold.
        T, train_nll = fit_temperature(
            design.y_true.values,
            design[prob_cols].to_numpy()
        )
        P_design = temperature_transform(design[prob_cols].to_numpy(), T)
        P_heldout = temperature_transform(heldout[prob_cols].to_numpy(), T)

        design_conf = P_design.max(axis=1)
        heldout_conf = P_heldout.max(axis=1)
        heldout_pred = np.asarray(CLASSES)[np.argmax(P_heldout, axis=1)]
        heldout_true = heldout.y_true.to_numpy()

        for cov in TARGET_COVERAGES:
            if cov >= 1.0:
                threshold = 0.0
            else:
                threshold = float(np.quantile(design_conf, 1.0 - cov))

            accepted = heldout_conf >= threshold
            n_acc = int(accepted.sum())

            fold_rows.append({
                'feature_set': fs,
                'model': model,
                'seed': int(seed),
                'outer_fold': int(fold),
                'calibration': 'temperature_crossfit_threshold_crossfit',
                'target_coverage': cov,
                'actual_coverage': n_acc / len(heldout),
                'rejection_rate': 1.0 - n_acc / len(heldout),
                'confidence_threshold': threshold,
                'temperature': T,
                'n_accepted': n_acc,
                'n_correct': int((heldout_true[accepted] == heldout_pred[accepted]).sum()) if n_acc else 0,
            })

            if n_acc:
                part = pd.DataFrame({
                    'feature_set': fs,
                    'model': model,
                    'seed': int(seed),
                    'outer_fold': int(fold),
                    'target_coverage': cov,
                    'y_true': heldout_true[accepted],
                    'y_pred': heldout_pred[accepted],
                })
                accepted_parts.append(part)

selective_by_fold = pd.DataFrame(fold_rows)
accepted_df = pd.concat(accepted_parts, ignore_index=True)

# Pool the five held-out folds within each seed after fold-external selection.
seed_rows = []
for (fs, model, seed, cov), g in selective_by_fold.groupby(
    ['feature_set', 'model', 'seed', 'target_coverage'], sort=True
):
    a = accepted_df[
        (accepted_df.feature_set == fs) &
        (accepted_df.model == model) &
        (accepted_df.seed == seed) &
        (accepted_df.target_coverage == cov)
    ]
    n_acc = int(len(a))
    total = int(g.shape[0] * N_TEST)

    seed_rows.append({
        'feature_set': fs,
        'model': model,
        'seed': int(seed),
        'calibration': 'temperature_crossfit_threshold_crossfit',
        'target_coverage': cov,
        'actual_coverage': n_acc / total,
        'rejection_rate': 1.0 - n_acc / total,
        'confidence_threshold': float(g.confidence_threshold.mean()),
        'accuracy_accepted': accuracy_score(a.y_true, a.y_pred) if n_acc else np.nan,
        'macro_f1_accepted': f1_score(
            a.y_true, a.y_pred, labels=CLASSES, average='macro', zero_division=0
        ) if n_acc else np.nan,
        'n_accepted': n_acc,
    })

selective = pd.DataFrame(seed_rows)
selective_by_fold.to_csv(OUT/'selective_classification_by_fold.csv', index=False)
selective.to_csv(OUT/'selective_classification_by_seed.csv', index=False)

selective_summary = (
    selective.groupby(['feature_set', 'model', 'target_coverage'])
    .agg(
        mean_actual_coverage=('actual_coverage', 'mean'),
        sd_actual_coverage=('actual_coverage', 'std'),
        mean_rejection_rate=('rejection_rate', 'mean'),
        mean_confidence_threshold=('confidence_threshold', 'mean'),
        mean_accuracy_accepted=('accuracy_accepted', 'mean'),
        sd_accuracy_accepted=('accuracy_accepted', 'std'),
        mean_macro_f1_accepted=('macro_f1_accepted', 'mean'),
        sd_macro_f1_accepted=('macro_f1_accepted', 'std')
    ).reset_index()
)
selective_summary.to_csv(TAB/'selective_classification_summary.csv', index=False)

display(
    selective_summary[
        (selective_summary.feature_set == 'FULL_11') &
        (selective_summary.model.isin(TOP_MODELS))
    ].sort_values(['model', 'target_coverage'], ascending=[True, False])
)


In [ ]:
plt.figure(figsize=(8,6))
for model in TOP_MODELS:
    z=selective_summary[
        (selective_summary.feature_set=='FULL_11') &
        (selective_summary.model==model)
    ].sort_values('mean_actual_coverage')
    plt.plot(z.mean_actual_coverage,z.mean_accuracy_accepted,marker='o',label=model)

plt.xlabel('Coverage')
plt.ylabel('Accuracy among accepted predictions')
plt.title('Selective classification — leading FULL_11 models')
plt.xlim(0.48,1.02)
plt.ylim(0.90,1.005)
plt.legend()
plt.tight_layout()
plt.savefig(FIG/'accuracy_coverage_top_FULL11_models.png',dpi=300,bbox_inches='tight')
plt.show()


## 10. Compact manuscript-ready summary tables

This section creates concise tables for later integration into the manuscript. It does not decide which model is “best” for deployment; it reports performance, inferential uncertainty, calibration, and selective-classification behavior.


In [ ]:
# Calibration summary restricted to FULL_11 and sorted by raw macro-F1 rank
if RUN_TEMPERATURE_SCALING:
    full_cal=cal_compare[cal_compare.feature_set=='FULL_11'].copy()
    full_cal=full_cal.merge(rank_full[['model','rank_macro_f1','mean_macro_f1']],on='model',how='left')
    full_cal=full_cal.sort_values(['rank_macro_f1','calibration'])
    full_cal.to_csv(TAB/'manuscript_full11_calibration_table.csv',index=False)
    display(full_cal)

# Key selective results at 100%, 90%, and 80% coverage
key_cov=selective_summary[
    (selective_summary.feature_set=='FULL_11') &
    (selective_summary.target_coverage.isin([1.0,0.9,0.8]))
].merge(
    rank_full[['model','rank_macro_f1','mean_macro_f1']],
    on='model',how='left'
).sort_values(['rank_macro_f1','target_coverage'],ascending=[True,False])

key_cov.to_csv(TAB/'manuscript_full11_selective_key_coverages.csv',index=False)
display(key_cov.head(30))


## 11. Integrity checks and run log


In [ ]:
# Core integrity checks
assert len(rank_full) == full_metrics.model.nunique()
assert metrics.groupby(['feature_set','model','seed','outer_fold']).size().eq(1).all()
assert oof.groupby(['feature_set','model','seed']).size().eq(4000).all()
assert set(TOP_MODELS).issubset(set(rank_full.model))
assert len(corr_tests) == len(rank_full)-1

if RUN_TEMPERATURE_SCALING:
    assert len(cal_oof) == len(oof)
    assert cal_oof.groupby(['feature_set','model','seed']).size().eq(4000).all()
    assert np.isfinite(cal_oof[['cal_'+c for c in prob_cols]].to_numpy()).all()

run_info={
    'notebook':'NB09_Statistical_Robustness_Calibration_Selective.ipynb',
    'base_model_retraining':False,
    'seeds':SEEDS,
    'outer_folds':OUTER_FOLDS,
    'n_train_per_outer_fold':N_TRAIN,
    'n_test_per_outer_fold':N_TEST,
    'corrected_resampling_test_train_ratio':TEST_TRAIN_RATIO,
    'reference_model_selected_by_mean_full11_macro_f1':REFERENCE_MODEL,
    'alpha':ALPHA,
    'primary_prespecified_equivalence_margin':PRIMARY_EQUIV_MARGIN,
    'equivalence_margin_sensitivity':EQUIV_MARGINS,
    'rope_margin_sensitivity':ROPE_MARGINS,
    'ece_bins':ECE_BINS,
    'temperature_scaling_crossfit':RUN_TEMPERATURE_SCALING,
    'temperature_scaling_protocol':'Within each model/feature-set/seed, each outer fold calibrated using OOF probabilities from the other four outer folds only.',
    'selective_coverages':TARGET_COVERAGES,
    'selective_threshold_protocol':'For each held-out outer fold, temperature and confidence threshold are estimated exclusively from the other four outer folds of the same seed and then applied to the held-out fold.',
    'note':'Repeated-CV folds are handled with corrected-resampling inference; repeated OOF predictions across seeds are not treated as independent biological samples.'
}
with open(LOG/'NB09_run_info.json','w') as f:
    json.dump(run_info,f,indent=2)

import sklearn, scipy, statsmodels
versions={
    'python':sys.version,
    'numpy':np.__version__,
    'pandas':pd.__version__,
    'scikit_learn':sklearn.__version__,
    'scipy':scipy.__version__,
    'statsmodels':statsmodels.__version__,
}
with open(LOG/'NB09_package_versions.json','w') as f:
    json.dump(versions,f,indent=2)

print('NB09 integrity checks PASSED.')
print('Reference model:',REFERENCE_MODEL)
print('Base models retrained: False')
print('Temperature scaling completed:',RUN_TEMPERATURE_SCALING)
print('NB09 completed successfully.')


### Main expected outputs

**Results**
- `03_RESULTS/NB09_ROBUSTNESS_CALIBRATION_SELECTIVE/all_metrics_harmonized.csv`
- `03_RESULTS/NB09_ROBUSTNESS_CALIBRATION_SELECTIVE/all_oof_harmonized.csv`
- `03_RESULTS/NB09_ROBUSTNESS_CALIBRATION_SELECTIVE/raw_probability_metrics_by_seed.csv`
- `03_RESULTS/NB09_ROBUSTNESS_CALIBRATION_SELECTIVE/crossfit_temperature_scaled_oof.csv`
- `03_RESULTS/NB09_ROBUSTNESS_CALIBRATION_SELECTIVE/temperature_scaled_probability_metrics_by_seed.csv`
- `03_RESULTS/NB09_ROBUSTNESS_CALIBRATION_SELECTIVE/selective_classification_by_fold.csv`
- `03_RESULTS/NB09_ROBUSTNESS_CALIBRATION_SELECTIVE/selective_classification_by_seed.csv`

**Tables**
- `full11_model_ranking.csv`
- `full11_corrected_resampled_tests_vs_reference.csv`
- `full11_equivalence_margin_sensitivity.csv`
- `full11_correlated_t_probability_summary.csv`
- `raw_calibration_summary.csv`
- `crossfit_temperature_by_fold.csv`
- `raw_vs_temperature_calibration_summary.csv`
- `selective_classification_summary.csv`
- manuscript-ready calibration and selective-classification tables

**Figures**
- reliability diagrams for the leading FULL_11 models
- `accuracy_coverage_top_FULL11_models.png`

After NB09 is complete, NB99 should be updated to include NB08 and NB09 outputs before creating the final reproducibility ZIP.
